In [1]:
import polars as pl
import glob
import os

In [2]:
saved_folder = r"C:\Users\ekadw\Documents\GitHub\DATA\streaming_in_polars"

In [3]:
data_jan_50 = pl.scan_parquet(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2009\yellow_taxi\yellow_tripdata_2009-01.parquet")
data_jan_50 = data_jan_50.head(50).collect()
parquet_file_jan = os.path.join(saved_folder, "data_jan_2009.parquet")
data_jan_50.write_parquet(parquet_file_jan)

data_feb_50 = pl.scan_parquet(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2009\yellow_taxi\yellow_tripdata_2009-02.parquet")
data_feb_50 = data_feb_50.head(50).collect()
parquet_file_feb = os.path.join(saved_folder, "data_feb_2009.parquet")
data_feb_50.write_parquet(parquet_file_feb)

data_mar_50 = pl.scan_parquet(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2009\yellow_taxi\yellow_tripdata_2009-03.parquet")
data_mar_50 = data_mar_50.head(50).collect()
parquet_file_mar = os.path.join(saved_folder, "data_mar_2009.parquet")
df.write_parquet(parquet_file_mar)

NameError: name 'df' is not defined

In [ ]:
files = glob.glob("C:/Users/ekadw/Documents/DATA/NY_Taxi/2009/yellow_taxi/yellow_tripdata_2009-*.parquet", recursive=True)
all_data = pl.scan_parquet(files)

In [ ]:
all_data = (
    all_data.select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
)

In [ ]:
import polars as pl
from sklearn.linear_model import SGDClassifier
import numpy as np

#yellow_2009 = pl.scan_parquet("yellow_2009/*.parquet")
clf = SGDClassifier(loss="log_loss")

batch_size = 100_000
start = 0

while True:
    df = yellow_2009.slice(start, batch_size).collect()
    if df.height == 0:
        break

    y = df["Tip_Category"].to_numpy()
    X = df.drop("Tip_Category").to_numpy()

    if start == 0:
        clf.partial_fit(X, y, classes=np.unique(y))
    else:
        clf.partial_fit(X, y)
    
    start += batch_size
